In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Convert timestamp to timezone-aware datetime in UTC then to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Total row counts
counts_df = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv'],
    'rows': [metric_df.shape[0], trace_df.shape[0]]
})

# 2) Unique cmdb_id values and counts per cmdb_id (top 20)
metric_cmdb_counts = metric_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
trace_cmdb_counts = trace_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# 3) Unique kpi_name and trace_name values with counts (show top 20 each)
kpi_counts = metric_df.groupby('kpi_name').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True).head(20)
trace_name_counts = trace_df.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True).head(20)

# 4) For each component-KPI pair in metric.csv compute global P95 and number of points
metric_grouped = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    p95_value=lambda x: np.nanpercentile(x.values, 95)
).reset_index().sort_values('count', ascending=False).reset_index(drop=True).head(20)

# 5) For each component-trace pair in trace.csv compute global P95 and number of points
trace_grouped = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count='count',
    p95_value=lambda x: np.nanpercentile(x.values, 95)
).reset_index().sort_values('count', ascending=False).reset_index(drop=True).head(20)

# 6) Filter data to incident window: April 11, 2020 04:30 to 05:00 (UTC+8) inclusive
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

metric_filtered = metric_df[(metric_df['timestamp_dt'] >= start_local) & (metric_df['timestamp_dt'] <= end_local)].copy()
trace_filtered = trace_df[(trace_df['timestamp_dt'] >= start_local) & (trace_df['timestamp_dt'] <= end_local)].copy()

# Select compact columns and limit to 20 rows
metric_filtered_display = metric_filtered[['timestamp_dt', 'cmdb_id', 'kpi_name', 'value']].sort_values('timestamp_dt').head(20)
trace_filtered_display = trace_filtered[['timestamp_dt', 'cmdb_id', 'trace_name', 'value']].sort_values('timestamp_dt').head(20)

# Final displayed variables (compact)
counts_df, metric_cmdb_counts.head(20), trace_cmdb_counts.head(20), kpi_counts, trace_name_counts, metric_grouped, trace_grouped, metric_filtered_display, trace_filtered_display

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry load and quick analysis:\n\n"
    "1) Row counts:\n"
    "- metric.csv: 91,115 rows\n"
    "- trace.csv: 6,240 rows\n\n"
    "2) Top cmdb_id counts (per file, top entries):\n"
    "- metric.csv top cmdb_ids (by row count, top 20 shown): db_003(2152), db_012(2097), db_013(2097), db_011(2096),\n"
    "  db_007(2094), db_001(2094), db_008(2093), db_002(2093), db_009(2093), db_005(2092), db_004(2092), db_006(2092),\n"
    "  db_010(2092), os_014(1993), os_009(1986), os_012(1985), os_010(1984), os_002(1984), os_011(1983), os_015(1982).\n"
    "- trace.csv top cmdb_ids (top entries): docker_001(840), docker_002(840), docker_003(840), docker_004(840),\n"
    "  docker_005(480), docker_006(480), docker_007(480), docker_008(480), os_021(480), os_022(480).\n\n"
    "3) KPI / trace feature cardinality (top names by frequency):\n"
    "- Top metric kpi_names (by count): Send_total, Disk_rd_ios, Disk_io_util, Outgoing_network_traffic,\n"
    "  Processor_load_5_min, Incoming_network_traffic, Memory_used, Memory_used_pct, Memory_available, Recv_total,\n"
    "  CPU_user_time, Page_pi, Disk_svctm, Page_po, Disk_wr_kbs, Disk_avgqu_sz, CPU_idle_pct, Disk_await, Agent_ping,\n"
    "  Disk_rd_kbs (these are the most frequent 20 KPIs).\n"
    "- Top trace names (by count): trace.self.jdbc.row_count, trace.self.jdbc.error_rate, trace.self.jdbc.duration_p95,\n"
    "  trace.self.jdbc.duration_mean, trace.self.local.row_count, trace.self.local.error_rate, trace.self.local.duration_p95,\n"
    "  trace.self.local.duration_mean, trace.self.csf.row_count, trace.self.csf.error_rate, trace.self.csf.duration_p95,\n"
    "  trace.self.csf.duration_mean, trace.self.flyremote.duration_mean, trace.self.flyremote.duration_p95,\n"
    "  trace.self.flyremote.error_rate, trace.self.flyremote.row_count, plus several trace.from_* remoteprocess series.\n\n"
    "4) Top component-KPI groups (global P95 and counts, top 20 by count):\n"
    "- The top groups by count shown include many with count ~61 (examples):\n"
    "  os_007 / Disk_rd_ios: count=61, p95=0.0\n"
    "  os_008 / Send_total: count=61, p95≈0.000685\n"
    "  os_022 / Outgoing_network_traffic: count=61, p95≈0.075096\n"
    "  os_015 / Incoming_network_traffic: count=61, p95≈0.002364\n"
    "  db_008 / CPU_free_pct: count=61, p95≈97.7573\n"
    "  (additional groups follow with counts ~60–61; see computed table for full top-20 values).\n\n"
    "5) Top component-trace groups (global P95 and counts, top 20 by count):\n"
    "- The top trace groups are mainly per-docker and include (examples):\n"
    "  docker_001 / trace.from_os_022.remoteprocess.duration_mean: count=30, p95≈4.4070\n"
    "  docker_001 / trace.from_os_022.remoteprocess.duration_p95: count=30, p95≈9.2301\n"
    "  docker_001 / trace.self.csf.duration_mean: count=30, p95≈1.0057\n"
    "  docker_001 / trace.self.jdbc.row_count: count=30, p95≈2343.25\n"
    "  (multiple other docker_* trace groups appear with count=30; see computed table for full top-20 values).\n\n"
    "6) Incident window data (filtered to 2020-04-11 04:30 to 05:00, UTC+8) — sample rows (up to 20 each):\n"
    "- metric samples (sorted by timestamp, first rows): examples at 2020-04-11 04:30+08:00 include\n"
    "  db_001 / ACS = 0.02, and many os_016 KPIs at 04:30 such as Outgoing_network_traffic=0.001197,\n"
    "  Memory_used_pct=26.38, Disk_wr_ios=8, Disk_svctm=1.12, etc. (20-row sample shown in results).\n"
    "- trace samples (at 2020-04-11 04:30+08:00 and others within window):\n"
    "  docker_001 / trace.from_os_022.remoteprocess.duration_mean = 0.665167,\n"
    "  docker_006 has multiple trace.self.jdbc.* and trace.self.local.* metrics at 04:30 (e.g. jdbc.row_count=1267),\n"
    "  docker_007/dockers show remoteprocess.* rows as well. (sample up to 20 rows shown in results).\n\n"
    "Notes / interpretation guidance:\n"
    "- Global P95 values were computed across the full files (no pre-filtering) as requested.\n"
    "- The metric dataset is heavily populated by DB node cmdb_ids (db_* entries) and many OS metrics; trace data is concentrated in docker_* components.\n"
    "- The incident window contains metric data primarily from os_016 and traces from several docker_* instances at 04:30.\n\n"
    "If you want, I can now: (a) produce the exact top-20 tables as CSV-like text, (b) compute deltas vs baseline P95 for the incident window,\n"
    "or (c) search for items in the window that exceed their global P95 thresholds."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file   rows
0  metric.csv  91115
1   trace.csv   6240,    cmdb_id  count
0   db_003   2152
1   db_012   2097
2   db_013   2097
3   db_011   2096
4   db_007   2094
5   db_001   2094
6   db_008   2093
7   db_002   2093
8   db_009   2093
9   db_005   2092
10  db_004   2092
11  db_006   2092
12  db_010   2092
13  os_014   1993
14  os_009   1986
15  os_012   1985
16  os_010   1984
17  os_002   1984
18  os_011   1983
19  os_015   1982,       cmdb_id  count
0  docker_001    840
1  docker_002    840
2  docker_003    840
3  docker_004    840
4  docker_005    480
5  docker_006    480
6  docker_007    480
7  docker_008    480
8      os_021    480
9      os_022    480,                     kpi_name  count
0                 Send_total   1323
1                Disk_rd_ios   1321
2               Disk_io_util   1321
3   Outgoing_network_traffic   1321
4       Processor_load_5_min   1321
5   Incoming_network_traffic   1321
6                Memory_used   1320
7            Memory_used_pct   1320
8           Memory_available   1320
9                 Recv_total   1319
10             CPU_user_time   1319
11                   Page_pi   1319
12                Disk_svctm   1319
13                   Page_po   1319
14               Disk_wr_kbs   1319
15             Disk_avgqu_sz   1319
16              CPU_idle_pct   1319
17                Disk_await   1319
18                Agent_ping   1319
19               Disk_rd_kbs   1318,                                            trace_name  count
0                           trace.self.jdbc.row_count    240
1                          trace.self.jdbc.error_rate    240
2                        trace.self.jdbc.duration_p95    240
3                       trace.self.jdbc.duration_mean    240
4                          trace.self.local.row_count    240
5                         trace.self.local.error_rate    240
6                       trace.self.local.duration_p95    240
7                      trace.self.local.duration_mean    240
8                            trace.self.csf.row_count    180
9                           trace.self.csf.error_rate    180
10                        trace.self.csf.duration_p95    180
11                       trace.self.csf.duration_mean    180
12                 trace.self.flyremote.duration_mean    120
13                  trace.self.flyremote.duration_p95    120
14                    trace.self.flyremote.error_rate    120
15                     trace.self.flyremote.row_count    120
16      trace.from_docker_003.remoteprocess.row_count     60
17     trace.from_docker_003.remoteprocess.error_rate     60
18   trace.from_docker_003.remoteprocess.duration_p95     60
19  trace.from_docker_003.remoteprocess.duration_mean     60,     cmdb_id                  kpi_name  count   p95_value
0    os_007               Disk_rd_ios     61    0.000000
1    os_008                Send_total     61    0.000685
2    os_022  Outgoing_network_traffic     61    0.075096
3    os_015  Incoming_network_traffic     61    0.002364
4    db_008              CPU_free_pct     61   97.757305
5    os_017                Send_total     61    0.296603
6    db_001              CPU_free_pct     61   99.018641
7    os_012              Disk_io_util     61    1.025076
8   osb_001           app.succee_rate     61    1.000000
9    os_006                Send_total     61    0.000471
10   db_003              CPU_free_pct     61   99.049704
11  osb_001                   app.num     61  554.000000
12   os_020              Disk_io_util     61    0.000000
13  osb_001            app.succee_num     61  554.000000
14   db_007             MEM_real_util     61   78.650000
15   os_012      Processor_load_5_min     61    0.015000
16  osb_001              app.avg_time     61    0.001892
17   os_018                Send_total     60    0.298961
18   os_019                Send_total     60    0.304893
19   os_004                   Page_pi     60    0.000000,        cmdb_id                                     trace_name  count    p95_value
0   docker_001  trace.from_os_022.remoteprocess.duration_mean     30     4.407002
1   docker_001   trace.from_os_022.remoteprocess.duration_p95     30     9.230137
2   docker_001     trace.from_os_022.remoteprocess.error_rate     30     0.000000
3   docker_001      trace.from_os_022.remoteprocess.row_count     30   146.300000
4   docker_001                   trace.self.csf.duration_mean     30     1.005676
5   docker_001                    trace.self.csf.duration_p95     30     2.960010
6   docker_001                      trace.self.csf.error_rate     30     0.000000
7   docker_001                       trace.self.csf.row_count     30   585.200000
8   docker_001             trace.self.flyremote.duration_mean     30     0.010581
9   docker_001              trace.self.flyremote.duration_p95     30     0.070567
10  docker_001                trace.self.flyremote.error_rate     30     0.000000
11  docker_001                 trace.self.flyremote.row_count     30   146.300000
12  docker_001                  trace.self.jdbc.duration_mean     30     0.002656
13  docker_001                   trace.self.jdbc.duration_p95     30     0.003000
14  docker_001                     trace.self.jdbc.error_rate     30     0.000000
15  docker_001                      trace.self.jdbc.row_count     30  2343.250000
16  docker_001                 trace.self.local.duration_mean     30     0.014410
17  docker_001                  trace.self.local.duration_p95     30     0.093467
18  docker_001                    trace.self.local.error_rate     30     0.000000
19  docker_001                     trace.self.local.row_count     30  2048.200000,                    timestamp_dt cmdb_id                  kpi_name       value
45627 2020-04-11 04:30:00+08:00  db_001                       ACS    0.020000
46634 2020-04-11 04:30:00+08:00  os_016  Outgoing_network_traffic    0.001197
46633 2020-04-11 04:30:00+08:00  os_016  Num_of_running_processes    1.000000
46632 2020-04-11 04:30:00+08:00  os_016          Num_of_processes  191.000000
46631 2020-04-11 04:30:00+08:00  os_016           Memory_used_pct   26.380000
46630 2020-04-11 04:30:00+08:00  os_016               Memory_used    5.131008
46629 2020-04-11 04:30:00+08:00  os_016               Memory_free   12.941925
46628 2020-04-11 04:30:00+08:00  os_016      Memory_available_pct   73.620000
46627 2020-04-11 04:30:00+08:00  os_016          Memory_available   14.320839
46626 2020-04-11 04:30:00+08:00  os_016  Incoming_network_traffic    0.000755
46625 2020-04-11 04:30:00+08:00  os_016                 ICMP_ping    1.000000
46624 2020-04-11 04:30:00+08:00  os_016               Disk_wr_kbs    0.059357
46623 2020-04-11 04:30:00+08:00  os_016               Disk_wr_ios    8.000000
46622 2020-04-11 04:30:00+08:00  os_016                Disk_svctm    1.120000
46621 2020-04-11 04:30:00+08:00  os_016               Disk_rd_kbs    0.000000
46620 2020-04-11 04:30:00+08:00  os_016               Disk_rd_ios    0.000000
46619 2020-04-11 04:30:00+08:00  os_016              Disk_io_util    0.000000
46618 2020-04-11 04:30:00+08:00  os_016                Disk_await    0.000000
46617 2020-04-11 04:30:00+08:00  os_016             Disk_avgqu_sz    0.011324
46616 2020-04-11 04:30:00+08:00  os_016                Cache_used    1.191616,                  timestamp_dt     cmdb_id                                         trace_name        value
0   2020-04-11 04:30:00+08:00  docker_001      trace.from_os_022.remoteprocess.duration_mean     0.665167
132 2020-04-11 04:30:00+08:00  docker_006  trace.from_docker_004.remoteprocess.duration_mean     0.027512
133 2020-04-11 04:30:00+08:00  docker_006   trace.from_docker_004.remoteprocess.duration_p95     0.040850
134 2020-04-11 04:30:00+08:00  docker_006     trace.from_docker_004.remoteprocess.error_rate     0.000000
135 2020-04-11 04:30:00+08:00  docker_006      trace.from_docker_004.remoteprocess.row_count   244.000000
136 2020-04-11 04:30:00+08:00  docker_006                      trace.self.jdbc.duration_mean     0.002478
137 2020-04-11 04:30:00+08:00  docker_006                       trace.self.jdbc.duration_p95     0.003000
138 2020-04-11 04:30:00+08:00  docker_006                         trace.self.jdbc.error_rate     0.000000
139 2020-04-11 04:30:00+08:00  docker_006                          trace.self.jdbc.row_count  1267.000000
140 2020-04-11 04:30:00+08:00  docker_006                     trace.self.local.duration_mean     0.006588
141 2020-04-11 04:30:00+08:00  docker_006                      trace.self.local.duration_p95     0.010000
131 2020-04-11 04:30:00+08:00  docker_006      trace.from_docker_003.remoteprocess.row_count   268.000000
142 2020-04-11 04:30:00+08:00  docker_006                        trace.self.local.error_rate     0.000000
144 2020-04-11 04:30:00+08:00  docker_007  trace.from_docker_001.remoteprocess.duration_mean     0.024351
145 2020-04-11 04:30:00+08:00  docker_007   trace.from_docker_001.remoteprocess.duration_p95     0.035000
146 2020-04-11 04:30:00+08:00  docker_007     trace.from_docker_001.remoteprocess.error_rate     0.000000
147 2020-04-11 04:30:00+08:00  docker_007      trace.from_docker_001.remoteprocess.row_count   251.000000
148 2020-04-11 04:30:00+08:00  docker_007  trace.from_docker_002.remoteprocess.duration_mean     0.024987
149 2020-04-11 04:30:00+08:00  docker_007   trace.from_docker_002.remoteprocess.duration_p95     0.036000
150 2020-04-11 04:30:00+08:00  docker_007     trace.from_docker_002.remoteprocess.error_rate     0.000000)```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse existing metric_df and tz if present; otherwise reload minimal needed items
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full metric_df
thresholds = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    global_p95=lambda x: np.nanpercentile(x.values, 95),
    global_p05=lambda x: np.nanpercentile(x.values, 5),
    total_points='count'
).reset_index()

# 2) Filter metric records to incident window (UTC+8): 2020-04-11 04:30 to 05:00 inclusive
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))
metric_window = metric_df[(metric_df['timestamp_dt'] >= start_local) & (metric_df['timestamp_dt'] <= end_local)].copy()

# 3) Mark anomalies: join thresholds then flag high/low
metric_win_th = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

def mark_anomaly(row):
    v = row['value']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if pd.isna(v) or pd.isna(p95) or pd.isna(p05):
        return None
    if v >= p95:
        return 'high'
    if v <= p05:
        return 'low'
    return None

metric_win_th['anomaly_type'] = metric_win_th.apply(mark_anomaly, axis=1)
anomalies = metric_win_th[metric_win_th['anomaly_type'].notnull()].copy()

# 4) Aggregate anomalies by (cmdb_id, kpi_name, anomaly_type)
agg = anomalies.groupby(['cmdb_id', 'kpi_name', 'anomaly_type']).agg(
    anomaly_count_in_window=('value', 'size'),
    earliest_anomaly_timestamp_dt=('timestamp_dt', 'min'),
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min'),
    global_p95=('global_p95', 'first'),
    global_p05=('global_p05', 'first')
).reset_index()

# compute severity: for high -> max_value / global_p95 ; for low -> global_p05 / min_value
def compute_sev(row):
    if row['anomaly_type'] == 'high':
        p95 = row['global_p95']
        maxv = row['max_value_in_window']
        if p95 == 0:
            return np.inf if maxv > 0 else 1.0
        return maxv / p95
    else:  # low
        p05 = row['global_p05']
        minv = row['min_value_in_window']
        if minv == 0:
            return np.inf if p05 > 0 else 1.0
        return p05 / minv

agg['severity'] = agg.apply(compute_sev, axis=1)

# sort by anomaly_count desc then severity desc, take top 20
agg_sorted = agg.sort_values(['anomaly_count_in_window', 'severity'], ascending=[False, False]).reset_index(drop=True).head(20)

# select and order required columns
agg_display = agg_sorted[[
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05', 'anomaly_type',
    'anomaly_count_in_window', 'earliest_anomaly_timestamp_dt',
    'max_value_in_window', 'min_value_in_window', 'severity'
]]

# 5) One-line summary naming top 1-3 candidate groups
top_n = agg_display.head(3)
if top_n.shape[0] == 0:
    summary = "No anomalies detected in the incident window for metric.csv based on global P95/P05."
else:
    parts = []
    for _, r in top_n.iterrows():
        parts.append(f"{r['cmdb_id']}/{r['kpi_name']} ({r['anomaly_type']}, count={int(r['anomaly_count_in_window'])}, sev={r['severity']:.2f})")
    summary = "Top candidates to inspect: " + "; ".join(parts)

# Display compact results: aggregated anomalies (top 20) and the one-line summary
agg_display, summary

```
Out[4]:
```
```python
summary = (
    "Anomaly detection results (metric.csv) — concise summary:\n\n"
    "What I did: computed global P95 and P05 per (cmdb_id, kpi_name) using the full metric.csv, "
    "filtered records to the incident window 2020-04-11 04:30–05:00 (UTC+8), and flagged values >=P95 as 'high' "
    "or <=P05 as 'low'. Then I aggregated anomalies by (cmdb_id, kpi_name).\n\n"
    "Top findings (by anomaly count and severity):\n"
    "1) os_007 / Disk_rd_ios — type: high, anomalies=31, earliest=2020-04-11 04:30+08:00, max=0.0, min=0.0, severity=1.00\n"
    "2) os_020 / Disk_io_util — type: high, anomalies=31, earliest=2020-04-11 04:30+08:00, max=0.0, min=0.0, severity=1.00\n"
    "3) osb_001 / app.succee_rate — type: high, anomalies=31, earliest=2020-04-11 04:30+08:00, max=1.0, min=1.0, severity=1.00\n\n"
    "Notable additional items:\n"
    "- db_003 / Sess_Used_Undo (high): anomalies=30, earliest=2020-04-11 04:30, max>0 while global P95=0 → very large severity (inf) — indicates values above a zero baseline.\n"
    "- Several OS and redis metrics show 30 anomalies in the window with varying severities (examples: Disk_io_util, Disk_await, connected_clients).\n"
    "- One low-type anomaly: db_011 / Sess_Connect — type: low, anomalies=30, earliest=2020-04-11 04:30, min=172, max=174, severity≈1.01 (slightly below global P05).\n\n"
    "Recommendation (next inspections): prioritize the top candidates: os_007/Disk_rd_ios, os_020/Disk_io_util, and osb_001/app.succee_rate. "
    "Also examine db_003/Sess_Used_Undo and db_011/Sess_Connect because their anomaly patterns (counts and severity) suggest meaningful deviation from baseline.\n"
    "If you want, I can now produce the full top-20 anomaly table, or show the raw anomaly rows from the incident window for any of these groups."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                   kpi_name  global_p95  global_p05 anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp_dt  max_value_in_window  min_value_in_window  severity
0      os_007                Disk_rd_ios    0.000000    0.000000         high                       31     2020-04-11 04:30:00+08:00             0.000000             0.000000  1.000000
1      os_020               Disk_io_util    0.000000    0.000000         high                       31     2020-04-11 04:30:00+08:00             0.000000             0.000000  1.000000
2     osb_001            app.succee_rate    1.000000    1.000000         high                       31     2020-04-11 04:30:00+08:00             1.000000             1.000000  1.000000
3      db_003             Sess_Used_Undo    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.015625             0.000000       inf
4      os_004               Disk_io_util    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.250000             0.000000       inf
5      os_006                 Disk_svctm    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             1.500000             0.000000       inf
6      os_020                Disk_rd_kbs    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.000130             0.000000       inf
7      os_020                    Page_pi    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.134868             0.000000       inf
8      os_022                 Disk_await    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.300000             0.000000       inf
9   redis_004  instantaneous_ops_per_sec    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             2.000000             0.000000       inf
10  redis_008  instantaneous_ops_per_sec    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             0.000000       inf
11  redis_004   total_commands_processed   26.000000   26.000000         high                       30     2020-04-11 04:30:00+08:00            65.000000            26.000000  2.500000
12     os_017                Disk_wr_ios    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             2.000000             1.000000  2.000000
13     os_006      Processor_load_15_min    0.008333    0.008333         high                       30     2020-04-11 04:30:00+08:00             0.010000             0.008333  1.200048
14  redis_008          connected_clients   23.000000   23.000000         high                       30     2020-04-11 04:30:00+08:00            24.000000            23.000000  1.043478
15  redis_012          connected_clients   24.000000   24.000000         high                       30     2020-04-11 04:30:00+08:00            25.000000            24.000000  1.041667
16  redis_004          connected_clients   25.000000   25.000000         high                       30     2020-04-11 04:30:00+08:00            26.000000            25.000000  1.040000
17     db_011               Sess_Connect  176.000000  174.000000          low                       30     2020-04-11 04:30:00+08:00           174.000000           172.000000  1.011628
18     db_004              Proc_Used_Pct    1.460000    1.460000         high                       30     2020-04-11 04:30:00+08:00             1.470000             1.460000  1.006849
19     db_004         Proc_User_Used_Pct    1.460000    1.460000         high                       30     2020-04-11 04:30:00+08:00             1.470000             1.460000  1.006849, 'Top candidates to inspect: os_007/Disk_rd_ios (high, count=31, sev=1.00); os_020/Disk_io_util (high, count=31, sev=1.00); osb_001/app.succee_rate (high, count=31, sev=1.00)')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing dataframes if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# Ensure global thresholds computed (use full metric_df)
try:
    thresholds
except NameError:
    thresholds = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

# Incident window (UTC+8)
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# Filter metric to window
metric_window = metric_df[(metric_df['timestamp_dt'] >= start_local) & (metric_df['timestamp_dt'] <= end_local)].copy()

# Candidate groups
candidates = [
    ('os_007', 'Disk_rd_ios'),
    ('os_020', 'Disk_io_util'),
    ('osb_001', 'app.succee_rate'),
    ('db_003', 'Sess_Used_Undo'),
    ('os_004', 'Disk_io_util'),
    ('os_006', 'Disk_svctm'),
    ('db_011', 'Sess_Connect')
]

cand_df = pd.DataFrame(candidates, columns=['cmdb_id','kpi_name'])

# Join thresholds to window, keep only candidate groups
win_th = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
# Filter to candidates
win_th = win_th.merge(cand_df, on=['cmdb_id','kpi_name'], how='inner')

# Mark anomalies using precomputed global_p95/global_p05
def mark_anom(v, p95, p05):
    if pd.isna(v) or pd.isna(p95) or pd.isna(p05):
        return None
    if v >= p95:
        return 'high'
    if v <= p05:
        return 'low'
    return None

win_th['anomaly_type'] = win_th.apply(lambda r: mark_anom(r['value'], r['global_p95'], r['global_p05']), axis=1)
anoms = win_th[win_th['anomaly_type'].notnull()].copy()

# Prepare seconds column for consecutive-minute grouping
anoms['ts_sec'] = anoms['timestamp_dt'].astype('int64') // 1_000_000_000

# Function to segment consecutive-minute runs per (cmdb_id,kpi_name)
segments = []
for (cmdb_id, kpi), grp in anoms.groupby(['cmdb_id','kpi_name']):
    g = grp.sort_values('ts_sec').reset_index(drop=True).copy()
    # compute diffs in seconds
    g['diff'] = g['ts_sec'].diff().fillna(0).astype(int)
    # start new segment when diff != 60
    g['segment_id'] = (g['diff'] != 60).cumsum()
    for seg_id, seg in g.groupby('segment_id'):
        seg_start_dt = seg['timestamp_dt'].min()
        seg_end_dt = seg['timestamp_dt'].max()
        length = seg.shape[0]
        maxv = seg['value'].max()
        minv = seg['value'].min()
        # choose threshold_type by mode of anomaly_type, fallback to first
        threshold_type = seg['anomaly_type'].mode().iloc[0] if not seg['anomaly_type'].mode().empty else seg['anomaly_type'].iloc[0]
        global_p95 = seg['global_p95'].iloc[0]
        global_p05 = seg['global_p05'].iloc[0]
        # compute severity_ratio
        if threshold_type == 'high':
            if global_p95 == 0:
                severity = np.inf if maxv > 0 else 1.0
            else:
                severity = maxv / global_p95
        else:  # low
            if minv == 0:
                severity = np.inf if global_p05 > 0 else 1.0
            else:
                severity = global_p05 / minv
        segments.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'threshold_type': threshold_type,
            'fault_start_timestamp_dt': seg_start_dt,
            'fault_end_timestamp_dt': seg_end_dt,
            'length_in_minutes': length,
            'max_value_in_segment': maxv,
            'min_value_in_segment': minv,
            'global_p95': global_p95,
            'global_p05': global_p05,
            'severity_ratio': severity
        })

segments_df = pd.DataFrame(segments)

# Separate faults (length >=2) and isolated spikes (length ==1)
if segments_df.empty:
    faults_df = pd.DataFrame(columns=['cmdb_id','kpi_name','threshold_type','fault_start_timestamp_dt','fault_end_timestamp_dt',
                                      'length_in_minutes','max_value_in_segment','min_value_in_segment','global_p95','global_p05','severity_ratio'])
    spikes_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_dt','value','threshold_type','global_p95','global_p05'])
else:
    faults_df = segments_df[segments_df['length_in_minutes'] >= 2].copy()
    spikes_segments = segments_df[segments_df['length_in_minutes'] == 1].copy()
    # For spikes, extract the single-point timestamp and value from anoms
    if not spikes_segments.empty:
        spikes_list = []
        for _, row in spikes_segments.iterrows():
            cmdb_id = row['cmdb_id']; kpi = row['kpi_name']; start = row['fault_start_timestamp_dt']
            rec = anoms[(anoms['cmdb_id']==cmdb_id)&(anoms['kpi_name']==kpi)&(anoms['timestamp_dt']==start)].iloc[0]
            spikes_list.append({
                'cmdb_id': cmdb_id,
                'kpi_name': kpi,
                'timestamp_dt': rec['timestamp_dt'],
                'value': rec['value'],
                'threshold_type': rec['anomaly_type'],
                'global_p95': rec['global_p95'],
                'global_p05': rec['global_p05']
            })
        spikes_df = pd.DataFrame(spikes_list)
    else:
        spikes_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp_dt','value','threshold_type','global_p95','global_p05'])

# Sort faults table by length desc then severity desc, limit 20
faults_display = faults_df.sort_values(['length_in_minutes','severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)
# For compactness, ensure selected columns and ordering
faults_display = faults_display[[
    'cmdb_id','kpi_name','threshold_type','fault_start_timestamp_dt','fault_end_timestamp_dt',
    'length_in_minutes','max_value_in_segment','min_value_in_segment','global_p95','global_p05','severity_ratio'
]]

# Spikes: show top single-point anomalies sorted by severity desc then earliest, limit 20
if not spikes_df.empty:
    spikes_display = spikes_df.sort_values(by=['timestamp_dt'], ascending=True).head(20).reset_index(drop=True)
else:
    spikes_display = spikes_df

# 7) One-line recommendation: single earliest fault segment from table A
if not faults_display.empty:
    earliest_row = faults_display.sort_values('fault_start_timestamp_dt').iloc[0]
    recommendation = (f"Inspect next: {earliest_row['cmdb_id']}/{earliest_row['kpi_name']} starting at "
                      f"{earliest_row['fault_start_timestamp_dt']}")
else:
    recommendation = "No multi-minute fault segments found among the candidates; no recommendation."

# Display compact tables and recommendation
faults_display, spikes_display, recommendation

```
Out[6]:
```
```python
summary = (
    "Fault identification summary (focused candidates):\n\n"
    "Findings:\n"
    "- Multi-minute fault segments (length >=2) detected for all 7 candidates:\n"
    "  1) os_007 / Disk_rd_ios — 31 minutes (2020-04-11 04:30 to 05:00+08:00), high anomalies, values all 0.0 (global P95=0.0).\n"
    "  2) os_020 / Disk_io_util — 31 minutes (04:30–05:00), high anomalies, values all 0.0 (global P95=0.0).\n"
    "  3) osb_001 / app.succee_rate — 31 minutes (04:30–05:00), high anomalies, values all 1.0 (global P95=1.0).\n"
    "  4) db_003 / Sess_Used_Undo — 30 minutes (04:30–04:59), high anomalies, max=0.015625, global P95=0.0 → severity infinite (above a zero baseline).\n"
    "  5) os_004 / Disk_io_util — 30 minutes (04:30–04:59), high anomalies, max=0.25, global P95=0.0 → severity infinite.\n"
    "  6) os_006 / Disk_svctm — 30 minutes (04:30–04:59), high anomalies, max=1.5, global P95=0.0 → severity infinite.\n"
    "  7) db_011 / Sess_Connect — 30 minutes (04:30–04:59), low anomalies, min=172, global P05=174, severity≈1.01 (slight drop below baseline).\n\n"
    "Isolated spikes:\n"
    "- None among these candidate groups (no single-point-only segments returned).\n\n"
    "Interpretation & next step:\n"
    "- Several OS and DB metrics show sustained anomalies through the incident window. For some KPIs the global P95/P05 is zero, so non-zero values produce very large (infinite) severity ratios — this indicates deviation from a zero baseline rather than an extreme multiplier.\n"
    "- Recommended immediate inspection: os_007 / Disk_rd_ios starting at 2020-04-11 04:30:00+08:00 (earliest multi-minute fault segment). After that, inspect os_020/Disk_io_util and osb_001/app.succee_rate, and the DB groups with infinite severity (db_003, os_004, os_006) to understand whether zero-baseline thresholds or measurement semantics explain the large severity values.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id         kpi_name threshold_type  fault_start_timestamp_dt    fault_end_timestamp_dt  length_in_minutes  max_value_in_segment  min_value_in_segment  global_p95  global_p05  severity_ratio
0   os_007      Disk_rd_ios           high 2020-04-11 04:30:00+08:00 2020-04-11 05:00:00+08:00                 31              0.000000                   0.0         0.0         0.0        1.000000
1   os_020     Disk_io_util           high 2020-04-11 04:30:00+08:00 2020-04-11 05:00:00+08:00                 31              0.000000                   0.0         0.0         0.0        1.000000
2  osb_001  app.succee_rate           high 2020-04-11 04:30:00+08:00 2020-04-11 05:00:00+08:00                 31              1.000000                   1.0         1.0         1.0        1.000000
3   db_003   Sess_Used_Undo           high 2020-04-11 04:30:00+08:00 2020-04-11 04:59:00+08:00                 30              0.015625                   0.0         0.0         0.0             inf
4   os_004     Disk_io_util           high 2020-04-11 04:30:00+08:00 2020-04-11 04:59:00+08:00                 30              0.250000                   0.0         0.0         0.0             inf
5   os_006       Disk_svctm           high 2020-04-11 04:30:00+08:00 2020-04-11 04:59:00+08:00                 30              1.500000                   0.0         0.0         0.0             inf
6   db_011     Sess_Connect            low 2020-04-11 04:30:00+08:00 2020-04-11 04:59:00+08:00                 30            174.000000                 172.0       176.0       174.0        1.011628, Empty DataFrame
Columns: [cmdb_id, kpi_name, timestamp_dt, value, threshold_type, global_p95, global_p05]
Index: [], 'Inspect next: os_007/Disk_rd_ios starting at 2020-04-11 04:30:00+08:00')```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full trace_df
trace_thresholds = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    global_p95=lambda x: np.nanpercentile(x.values, 95),
    global_p05=lambda x: np.nanpercentile(x.values, 5),
    total_points='count'
).reset_index()

# 2) Filter trace records to incident window (UTC+8)
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))
trace_window = trace_df[(trace_df['timestamp_dt'] >= start_local) & (trace_df['timestamp_dt'] <= end_local)].copy()

# 3) Mark anomalies by joining thresholds
trace_win_th = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

def mark_anomaly_trace(v, p95, p05):
    if pd.isna(v) or pd.isna(p95) or pd.isna(p05):
        return None
    if v >= p95:
        return 'high'
    if v <= p05:
        return 'low'
    return None

trace_win_th['anomaly_type'] = trace_win_th.apply(lambda r: mark_anomaly_trace(r['value'], r['global_p95'], r['global_p05']), axis=1)
trace_anoms = trace_win_th[trace_win_th['anomaly_type'].notnull()].copy()

# 4) Aggregate anomalies by (cmdb_id, trace_name)
if trace_anoms.empty:
    agg_display = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_metric'
    ])
else:
    agg = trace_anoms.groupby(['cmdb_id', 'trace_name', 'anomaly_type']).agg(
        anomaly_count_in_window=('value', 'size'),
        earliest_anomaly_timestamp_dt=('timestamp_dt', 'min'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        global_p95=('global_p95', 'first'),
        global_p05=('global_p05', 'first')
    ).reset_index()

    # compute severity_metric
    def sev(row):
        if row['anomaly_type'] == 'high':
            p95 = row['global_p95']
            maxv = row['max_value_in_window']
            if p95 == 0:
                return np.inf if maxv > 0 else 1.0
            return maxv / p95
        else:
            p05 = row['global_p05']
            minv = row['min_value_in_window']
            if minv == 0:
                return np.inf if p05 > 0 else 1.0
            return p05 / minv

    agg['severity_metric'] = agg.apply(sev, axis=1)

    # sort by count desc then severity desc and take top 20
    agg_display = agg.sort_values(['anomaly_count_in_window', 'severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)

    # select required columns and order
    agg_display = agg_display[[
        'cmdb_id','trace_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_metric'
    ]]

# 5) One-line summary: top 1-3 trace groups whose earliest_anomaly_timestamp_dt == start_local
if agg_display.shape[0] == 0:
    summary = "No trace anomalies detected in the incident window."
else:
    coincident = agg_display[agg_display['earliest_anomaly_timestamp_dt'] == start_local].copy()
    if coincident.empty:
        summary = "No trace anomaly groups start exactly at 2020-04-11 04:30:00+08:00."
    else:
        top_coincident = coincident.sort_values(['anomaly_count_in_window','severity_metric'], ascending=[False, False]).head(3)
        parts = []
        for _, r in top_coincident.iterrows():
            cm = r['cmdb_id']
            level = ('container-level' if str(cm).startswith('docker_') else
                     'service/node-level' if (str(cm).startswith('db_') or str(cm).startswith('os_')) else 'other-level')
            parts.append(f"{cm}/{r['trace_name']} ({level})")
        summary = "Top coincident trace groups at 04:30+08: " + "; ".join(parts)

# Display compact aggregated top-20 groups and the one-line summary
agg_display, summary

```
Out[7]:
```
```python
summary = (
    "Trace-level anomaly detection (concise):\n\n"
    "What was done: computed global P95/P05 per (cmdb_id, trace_name) from the full trace.csv, "
    "filtered to the incident window 2020-04-11 04:30–05:00 (UTC+8), and flagged values >=P95 as 'high' or <=P05 as 'low'.\n\n"
    "Top aggregated trace anomalies (examples from the top-20):\n"
    "- docker_005 / trace.self.jdbc.duration_p95 — high, count=30, earliest=2020-04-11 04:30+08:00, max=0.011, global_p95=0.003, severity≈3.67\n"
    "- docker_006 / trace.self.jdbc.duration_p95 — high, count=30, earliest=2020-04-11 04:30+08:00, max=0.011, global_p95=0.003, severity≈3.67\n"
    "- docker_007 / trace.self.jdbc.duration_p95 — high, count=30, earliest=2020-04-11 04:30+08:00, max=0.011, global_p95=0.003, severity≈3.67\n"
    "Plus many docker_* groups showing sustained high error_rate or other trace flags at 04:30 with count=30 (most error_rate series have zero baselines).\n\n"
    "Coincident groups to inspect (earliest anomaly == 2020-04-11 04:30+08:00):\n"
    "- docker_005/trace.self.jdbc.duration_p95 (container-level)\n"
    "- docker_006/trace.self.jdbc.duration_p95 (container-level)\n"
    "- docker_007/trace.self.jdbc.duration_p95 (container-level)\n\n"
    "Interpretation and recommendation:\n"
    "- These top coincident trace anomalies are at the container level (docker_*). Their JDBC duration_p95 is ~3.7x the global P95, starting exactly at 04:30+08:00, matching metric-level faults in the same window.\n"
    "- Also investigate docker-level error_rate groups (many show high flags at 04:30) but note many have zero baselines—interpret zeros carefully.\n"
    "- Next step: perform trace-level root-cause tracing on docker_005/006/007 around 2020-04-11 04:30+08:00 (follow JDBC spans and upstream/downstream remoteprocess traces) to determine whether the issue is container-local (CPU/IO) or a downstream service/db problem."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name  global_p95  global_p05 anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp_dt  max_value_in_window  min_value_in_window  severity_metric
0   docker_005                  trace.self.jdbc.duration_p95       0.003       0.003         high                       30     2020-04-11 04:30:00+08:00                0.011                0.003         3.666667
1   docker_006                  trace.self.jdbc.duration_p95       0.003       0.003         high                       30     2020-04-11 04:30:00+08:00                0.011                0.003         3.666667
2   docker_007                  trace.self.jdbc.duration_p95       0.003       0.003         high                       30     2020-04-11 04:30:00+08:00                0.011                0.003         3.666667
3   docker_001    trace.from_os_022.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
4   docker_001                     trace.self.csf.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
5   docker_001               trace.self.flyremote.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
6   docker_001                  trace.self.jdbc.duration_p95       0.003       0.003         high                       30     2020-04-11 04:30:00+08:00                0.003                0.003         1.000000
7   docker_001                    trace.self.jdbc.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
8   docker_001                   trace.self.local.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
9   docker_001  trace.to_docker_007.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
10  docker_001  trace.to_docker_008.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
11  docker_002    trace.from_os_022.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
12  docker_002                     trace.self.csf.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
13  docker_002               trace.self.flyremote.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
14  docker_002                    trace.self.jdbc.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
15  docker_002                   trace.self.local.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
16  docker_002  trace.to_docker_007.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
17  docker_002  trace.to_docker_008.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
18  docker_003    trace.from_os_021.remoteprocess.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000
19  docker_003                     trace.self.csf.error_rate       0.000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000         1.000000, 'Top coincident trace groups at 04:30+08: docker_005/trace.self.jdbc.duration_p95 (container-level); docker_006/trace.self.jdbc.duration_p95 (container-level); docker_007/trace.self.jdbc.duration_p95 (container-level)')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse trace_df and trace_thresholds if available; otherwise load/compute
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace_thresholds
except NameError:
    trace_thresholds = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# Filter to window and chosen cmdb_ids
cmdb_filter = ['docker_005', 'docker_006', 'docker_007']
trace_filt = trace_df[
    (trace_df['timestamp_dt'] >= start_local) &
    (trace_df['timestamp_dt'] <= end_local) &
    (trace_df['cmdb_id'].isin(cmdb_filter))
].copy()

# 1) Up to 50 rows of raw filtered traces sorted by timestamp
raw_display = trace_filt.sort_values('timestamp_dt')[['timestamp_dt','cmdb_id','trace_name','value']].head(50)

# 2) Aggregate counts by trace_name for these three cmdb_ids; return groups sorted by count desc (limit 50)
group_counts_trace = trace_filt.groupby('trace_name').size().reset_index(name='count').sort_values('count', ascending=False).head(50)

# 3) From those groups, list trace_name values containing specified substrings and compute stats per (cmdb_id, trace_name)
pattern = r'remoteprocess|to_|from_|jdbc'
matching = trace_filt[trace_filt['trace_name'].str.contains(pattern, regex=True, na=False)].copy()

if matching.empty:
    matching_groups_display = pd.DataFrame(columns=[
        'cmdb_id','trace_name','count_in_window','earliest_timestamp_dt','max_value_in_window','global_p95','severity'
    ])
else:
    agg = matching.groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max')
    ).reset_index()

    # join global_p95 from precomputed thresholds
    agg = agg.merge(trace_thresholds[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')

    # compute severity = max_value_in_window / global_p95 (if global_p95>0 else inf)
    def compute_sev(row):
        p95 = row['global_p95']
        maxv = row['max_value_in_window']
        if pd.isna(p95) or p95 == 0:
            return np.inf if (not pd.isna(maxv) and maxv > 0) else np.nan
        return maxv / p95

    agg['severity'] = agg.apply(compute_sev, axis=1)

    # sort by count desc then severity desc and limit to 20 for compactness
    matching_groups_display = agg.sort_values(['count_in_window','severity'], ascending=[False, False]).head(20)

# Display results (compact)
raw_display, group_counts_trace, matching_groups_display

```
Out[8]:
```
```python
summary = (
    "Filtered traces (docker_005, docker_006, docker_007) for 2020-04-11 04:30–05:00+08:00 — concise findings:\n\n"
    "1) Raw sample: up to 50 rows were returned (sorted by timestamp). Examples: earliest rows at 04:30 include\n"
    "   - docker_005 / trace.from_docker_003.remoteprocess.duration_mean = 0.029147\n"
    "   - docker_006 / trace.self.jdbc.row_count = 1267\n"
    "   - docker_007 / trace.from_docker_001.remoteprocess.duration_p95 = 0.047800\n\n"
    "2) Counts by trace_name (top groups):\n"
    "   - The most frequent trace_names (count=90 across the 3 containers) are: trace.self.jdbc.duration_mean, trace.self.jdbc.duration_p95, trace.self.jdbc.error_rate, trace.self.jdbc.row_count, trace.self.local.duration_mean, trace.self.local.duration_p95.\n"
    "   - Many remoteprocess.* groups appear with count=30 per specific from/to remote traces (these are the per-remote-source/destination series).\n\n"
    "3) Substring-matching groups (remoteprocess / to_ / from_ / jdbc) — notable statistics:\n"
    "   - docker_005 / trace.self.jdbc.duration_p95: count=30, earliest=2020-04-11 04:30+08:00, max=0.011, global_p95=0.003, severity≈3.67\n"
    "   - docker_006 / trace.self.jdbc.duration_p95: same stats as above (count=30, severity≈3.67)\n"
    "   - docker_007 / trace.self.jdbc.duration_p95: same stats as above (count=30, severity≈3.67)\n"
    "   - jdbc.duration_mean for docker_005/006/007: count=30 each, max≈0.0051–0.0053, global_p95≈0.00256–0.00263, severity≈2.0\n"
    "   - Several trace.from_docker_*/trace.to_* remoteprocess.duration_p95 and duration_mean groups show count=30 with moderate severity (≈1.3–1.6), earliest=04:30, e.g. docker_006 trace.from_docker_004.duration_p95 (max≈0.07335, global_p95≈0.05556, sev≈1.32).\n"
    "   - docker_005 / trace.self.jdbc.row_count: count=30, max=1565, global_p95=1442.15, severity≈1.09.\n\n"
    "Interpretation & recommendation:\n"
    "- Container-level JDBC tail latency (trace.self.jdbc.duration_p95) is the top anomaly for docker_005/006/007 (≈3.7x P95) starting exactly at 04:30+08:00, coinciding with metric faults.\n"
    "- Remoteprocess traces between containers also show elevated durations (severity ~1.3–1.6) at the same start time—suggests cross-container latency impact.\n"
    "- Prioritize trace inspection of docker_005/006/007 JDBC spans and their remoteprocess edges around 2020-04-11 04:30+08:00 to determine whether the root cause is container-local (CPU/IO) or a downstream service dependency."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 timestamp_dt     cmdb_id                                         trace_name        value
112 2020-04-11 04:30:00+08:00  docker_005  trace.from_docker_003.remoteprocess.duration_mean     0.029147
138 2020-04-11 04:30:00+08:00  docker_006                         trace.self.jdbc.error_rate     0.000000
139 2020-04-11 04:30:00+08:00  docker_006                          trace.self.jdbc.row_count  1267.000000
140 2020-04-11 04:30:00+08:00  docker_006                     trace.self.local.duration_mean     0.006588
141 2020-04-11 04:30:00+08:00  docker_006                      trace.self.local.duration_p95     0.010000
..                        ...         ...                                                ...          ...
132 2020-04-11 04:30:00+08:00  docker_006  trace.from_docker_004.remoteprocess.duration_mean     0.027512
133 2020-04-11 04:30:00+08:00  docker_006   trace.from_docker_004.remoteprocess.duration_p95     0.040850
124 2020-04-11 04:30:00+08:00  docker_005                     trace.self.local.duration_mean     0.007252
345 2020-04-11 04:31:00+08:00  docker_006                       trace.self.jdbc.duration_p95     0.003000
353 2020-04-11 04:31:00+08:00  docker_007   trace.from_docker_001.remoteprocess.duration_p95     0.047800

[50 rows x 4 columns],                                            trace_name  count
16                      trace.self.jdbc.duration_mean     90
17                       trace.self.jdbc.duration_p95     90
18                         trace.self.jdbc.error_rate     90
19                          trace.self.jdbc.row_count     90
20                     trace.self.local.duration_mean     90
..                                                ...    ...
4   trace.from_docker_002.remoteprocess.duration_mean     30
3       trace.from_docker_001.remoteprocess.row_count     30
2      trace.from_docker_001.remoteprocess.error_rate     30
1    trace.from_docker_001.remoteprocess.duration_p95     30
0   trace.from_docker_001.remoteprocess.duration_mean     30

[24 rows x 2 columns],        cmdb_id                                         trace_name  count_in_window     earliest_timestamp_dt  max_value_in_window   global_p95  severity
9   docker_005                       trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
21  docker_006                       trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
33  docker_007                       trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
32  docker_007                      trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005320     0.002627  2.025471
20  docker_006                      trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005250     0.002621  2.003129
8   docker_005                      trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005134     0.002563  2.003043
13  docker_006   trace.from_docker_003.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067200     0.042550  1.579318
5   docker_005   trace.from_docker_004.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067700     0.044440  1.523402
1   docker_005   trace.from_docker_003.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000     0.044357  1.465367
25  docker_007   trace.from_docker_001.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000     0.045640  1.424189
24  docker_007  trace.from_docker_001.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039240     0.027561  1.423769
4   docker_005  trace.from_docker_004.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.041514     0.029196  1.421907
29  docker_007   trace.from_docker_002.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.064000     0.045027  1.421354
28  docker_007  trace.from_docker_002.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.038793     0.027519  1.409658
16  docker_006  trace.from_docker_004.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039329     0.028636  1.373428
12  docker_006  trace.from_docker_003.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.037523     0.028129  1.333976
17  docker_006   trace.from_docker_004.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.073350     0.055560  1.320194
0   docker_005  trace.from_docker_003.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.036743     0.028896  1.271545
11  docker_005                          trace.self.jdbc.row_count               30 2020-04-11 04:30:00+08:00          1565.000000  1442.150000  1.085185
3   docker_005      trace.from_docker_003.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           327.000000   308.400000  1.060311)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# Reuse existing metric_df and thresholds if available; otherwise load/compute
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

try:
    thresholds
except NameError:
    thresholds = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

# Incident window (UTC+8)
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# 2) Filter metric to window and restrict to cmdb_id starting with 'db_'
metric_window = metric_df[(metric_df['timestamp_dt'] >= start_local) & (metric_df['timestamp_dt'] <= end_local)].copy()
metric_window_db = metric_window[metric_window['cmdb_id'].str.startswith('db_')].copy()

# 2b) Join precomputed thresholds (from full file) to windowed db metrics
metric_db_th = metric_window_db.merge(
    thresholds[['cmdb_id','kpi_name','global_p95','global_p05']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# 2c) Mark anomalies based on global thresholds
def mark_anom_type(row):
    v = row['value']
    p95 = row['global_p95']
    p05 = row['global_p05']
    if pd.isna(v) or (pd.isna(p95) and pd.isna(p05)):
        return None
    if (not pd.isna(p95)) and (v >= p95):
        return 'high'
    if (not pd.isna(p05)) and (v <= p05):
        return 'low'
    return None

metric_db_th['anomaly_type'] = metric_db_th.apply(mark_anom_type, axis=1)
anoms_db = metric_db_th[metric_db_th['anomaly_type'].notnull()].copy()

# 3) Aggregate anomalies by (cmdb_id, kpi_name, anomaly_type)
if anoms_db.empty:
    agg_display = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_ratio'
    ])
else:
    agg = anoms_db.groupby(['cmdb_id','kpi_name','anomaly_type']).agg(
        anomaly_count_in_window=('value','size'),
        earliest_anomaly_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_p95=('global_p95','first'),
        global_p05=('global_p05','first')
    ).reset_index()

    # compute severity_ratio with division-by-zero -> inf
    def compute_sev(row):
        if row['anomaly_type'] == 'high':
            denom = row['global_p95']
            num = row['max_value_in_window']
            if pd.isna(denom) or denom == 0:
                return np.inf if (not pd.isna(num) and num > 0) else np.nan
            return num / denom
        else:  # low
            denom = row['min_value_in_window']
            num = row['global_p05']
            if pd.isna(denom) or denom == 0:
                return np.inf if (not pd.isna(num) and num > 0) else np.nan
            return num / denom

    agg['severity_ratio'] = agg.apply(compute_sev, axis=1)

    # sort by anomaly_count desc then severity desc, limit 20
    agg_display = agg.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

    # select required columns
    agg_display = agg_display[[
        'cmdb_id','kpi_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_ratio'
    ]]

# 4) One-line summary: top 1-3 db_* groups whose earliest_anomaly_timestamp_dt == start_local
if agg_display.shape[0] == 0:
    summary = "No db_* anomalies detected in the incident window."
else:
    coincident = agg_display[agg_display['earliest_anomaly_timestamp_dt'] == start_local].copy()
    if coincident.empty:
        summary = "No db_* anomaly groups start exactly at 2020-04-11 04:30:00+08:00."
    else:
        top3 = coincident.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(3)
        parts = []
        for _, r in top3.iterrows():
            parts.append(f"{r['cmdb_id']}/{r['kpi_name']} ({r['anomaly_type']}, count={int(r['anomaly_count_in_window'])}, sev={np.round(r['severity_ratio'],3) if not pd.isna(r['severity_ratio']) else 'nan'})")
        summary = "Top db_* groups coincident at 04:30+08: " + "; ".join(parts)

# Display compact aggregated anomalies and one-line summary
agg_display, summary

```
Out[9]:
```
```python
summary = (
    "DB-focused anomaly detection (metrics) — concise summary:\n\n"
    "What was done: used global P95/P05 computed from the full metric.csv, filtered metrics to the incident window "
    "2020-04-11 04:30–05:00 (UTC+8), flagged values >=P95 as 'high' and <=P05 as 'low', and aggregated anomalies for cmdb_id starting with 'db_'.\n\n"
    "Top findings (sorted by anomaly count then severity, all earliest anomalies at 2020-04-11 04:30+08:00):\n"
    "- db_003 / Sess_Used_Undo — type: high, anomalies=30, earliest=2020-04-11 04:30, max=0.015625, global_p95=0.0 → severity=inf (non-zero above zero baseline).\n"
    "- db_011 / Sess_Connect — type: low, anomalies=30, earliest=2020-04-11 04:30, min=172, global_p05=174 → severity≈1.01 (slight drop below baseline).\n"
    "- db_004 / Proc_Used_Pct (and Proc_User_Used_Pct) — type: high, anomalies=30, earliest=2020-04-11 04:30, small uplift (severity≈1.007).\n\n"
    "Additional notes: many db_* KPIs show sustained 30-minute anomaly runs (04:30–04:59) with global thresholds equal to constant values for some KPIs (e.g., Hang/On_Off_State/MEM_Total), producing anomaly flags but neutral severity=1.0. Metrics with global_p95==0 produce infinite severity when any positive value appears (interpret as deviation from a zero baseline rather than a multiplicative spike).\n\n"
    "Recommendation (next inspection): prioritize investigation of db_003/Sess_Used_Undo (high, infinite severity), db_011/Sess_Connect (low, connection drop), and db_004/Proc_Used_Pct (small CPU/process usage uptick) to correlate with the container-level JDBC anomalies seen on docker_005/006/007 at 04:30+08:00."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id            kpi_name  global_p95  global_p05 anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp_dt  max_value_in_window  min_value_in_window  severity_ratio
0   db_003      Sess_Used_Undo    0.000000    0.000000         high                       30     2020-04-11 04:30:00+08:00             0.015625             0.000000             inf
1   db_011        Sess_Connect  176.000000  174.000000          low                       30     2020-04-11 04:30:00+08:00           174.000000           172.000000        1.011628
2   db_004       Proc_Used_Pct    1.460000    1.460000         high                       30     2020-04-11 04:30:00+08:00             1.470000             1.460000        1.006849
3   db_004  Proc_User_Used_Pct    1.460000    1.460000         high                       30     2020-04-11 04:30:00+08:00             1.470000             1.460000        1.006849
4   db_001                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
5   db_001           MEM_Total  381.902264  381.902264         high                       30     2020-04-11 04:30:00+08:00           381.902264           381.902264        1.000000
6   db_001        On_Off_State    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
7   db_002                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
8   db_002           MEM_Total  381.902264  381.902264         high                       30     2020-04-11 04:30:00+08:00           381.902264           381.902264        1.000000
9   db_002        On_Off_State    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
10  db_003                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
11  db_003           MEM_Total  381.902264  381.902264         high                       30     2020-04-11 04:30:00+08:00           381.902264           381.902264        1.000000
12  db_003        On_Off_State    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
13  db_004                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
14  db_004           MEM_Total  381.902264  381.902264         high                       30     2020-04-11 04:30:00+08:00           381.902264           381.902264        1.000000
15  db_004        On_Off_State    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
16  db_005                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
17  db_005           MEM_Total  506.929929  506.929929         high                       30     2020-04-11 04:30:00+08:00           506.929929           506.929929        1.000000
18  db_005        On_Off_State    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000
19  db_006                Hang    1.000000    1.000000         high                       30     2020-04-11 04:30:00+08:00             1.000000             1.000000        1.000000, 'Top db_* groups coincident at 04:30+08: db_003/Sess_Used_Undo (high, count=30, sev=inf); db_011/Sess_Connect (low, count=30, sev=1.012); db_004/Proc_Used_Pct (high, count=30, sev=1.007)')```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
import re
from datetime import datetime

# reuse existing trace_df and trace_thresholds if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace_thresholds
except NameError:
    trace_thresholds = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# filter to incident window
trace_window = trace_df[(trace_df['timestamp_dt'] >= start_local) & (trace_df['timestamp_dt'] <= end_local)].copy()

# 2) Aggregate by trace_name across all cmdb_ids for trace_name containing db-related substrings (case-insensitive)
db_pattern = r'db'  # as requested: 'db' or 'DB' or 'to_db' or 'from_db' or '_db_' or '.db.' -> case-insensitive 'db' matches these
mask_db = trace_window['trace_name'].str.contains(db_pattern, case=False, na=False)

agg_by_trace = trace_window.groupby('trace_name').agg(
    total_count_in_window=('value','size'),
    earliest_timestamp_dt=('timestamp_dt','min'),
    max_value_in_window=('value','max')
).reset_index()

# attach a representative global_p95 for the trace_name across cmdb_ids (use max global_p95 among cmdb_ids for that trace_name)
global_p95_by_trace = trace_thresholds.groupby('trace_name')['global_p95'].max().reset_index().rename(columns={'global_p95':'global_p95'})
agg_by_trace = agg_by_trace.merge(global_p95_by_trace, on='trace_name', how='left')

# select only db-related trace_name groups
db_groups = agg_by_trace[agg_by_trace['trace_name'].str.contains(db_pattern, case=False, na=False)].copy()

# compute severity
def sev_row(r):
    p95 = r['global_p95']
    mv = r['max_value_in_window']
    if pd.isna(p95) or p95 == 0:
        return np.inf if (not pd.isna(mv) and mv > 0) else np.nan
    return mv / p95

if not db_groups.empty:
    db_groups['severity'] = db_groups.apply(sev_row, axis=1)
    db_groups_display = db_groups.sort_values(['total_count_in_window','severity'], ascending=[False,False]).head(20)
else:
    db_groups_display = pd.DataFrame(columns=['trace_name','total_count_in_window','earliest_timestamp_dt','max_value_in_window','global_p95','severity'])

# 3) For docker_005/006/007 list distinct trace_name values that match same substrings and counts
docker_list = ['docker_005','docker_006','docker_007']
trace_dockers = trace_window[trace_window['cmdb_id'].isin(docker_list)].copy()
dock_mask = trace_dockers['trace_name'].str.contains(db_pattern, case=False, na=False)
docker_matches = trace_dockers[dock_mask].groupby(['cmdb_id','trace_name']).agg(
    count_in_window=('value','size'),
    earliest_timestamp_dt=('timestamp_dt','min'),
    max_value_in_window=('value','max')
).reset_index()

# join global_p95 per (cmdb_id, trace_name)
docker_matches = docker_matches.merge(trace_thresholds[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
docker_matches['severity'] = docker_matches.apply(lambda r: np.inf if (pd.isna(r['global_p95']) or r['global_p95']==0) and r['max_value_in_window']>0 else (r['max_value_in_window']/r['global_p95'] if r['global_p95'] and r['global_p95']>0 else np.nan), axis=1)
docker_matches_display = docker_matches.sort_values(['count_in_window','severity'], ascending=[False,False]).head(50)

# 4) If no direct db-identifying trace_name found, fallback: top 10 trace.from_* or trace.to_* remoteprocess groups
if db_groups_display.shape[0] == 0:
    # pattern for remoteprocess/from/to
    rp_pattern = r'trace\.from_|trace\.to_|remoteprocess'
    remote_mask = trace_window['trace_name'].str.contains(rp_pattern, case=False, na=False)
    remote_groups = trace_window[remote_mask].groupby(['cmdb_id','trace_name']).agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max')
    ).reset_index()
    # join global_p95
    remote_groups = remote_groups.merge(trace_thresholds[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
    remote_groups['contains_db'] = remote_groups['trace_name'].str.contains(db_pattern, case=False, na=False)
    # severity
    remote_groups['severity'] = remote_groups.apply(lambda r: (np.inf if (pd.isna(r['global_p95']) or r['global_p95']==0) and r['max_value_in_window']>0 else (r['max_value_in_window']/r['global_p95'] if r['global_p95'] and r['global_p95']>0 else np.nan)), axis=1)
    fallback_display = remote_groups.sort_values(['count_in_window','severity'], ascending=[False,False]).head(10)
else:
    fallback_display = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','earliest_timestamp_dt','max_value_in_window','global_p95','contains_db','severity'])

# 5) One-line summary whether any trace group explicitly references db_* services and which ones
if db_groups_display.shape[0] > 0:
    # list top matches (trace_name and total_count)
    top_matches = db_groups_display[['trace_name','total_count_in_window']].head(10)
    pairs = [f"{row.trace_name} (count={int(row.total_count_in_window)})" for _, row in top_matches.iterrows()]
    summary = "Found trace_name groups referencing 'db' substrings: " + "; ".join(pairs)
else:
    summary = "No trace_name explicitly referencing db_* found in the window; fallback remoteprocess/to_/from_ groups listed."

# final compact outputs
db_groups_display, docker_matches_display, fallback_display, summary

```
Out[10]:
```
```python
summary = (
    "Trace search for DB references (incident window 2020-04-11 04:30–05:00+08:00):\n\n"
    "Findings:\n"
    "- No trace_name strings explicitly contain a db_* component identifier (e.g., 'db_123') in the window.\n"
    "- However, many JDBC-related trace groups are present and widespread: trace.self.jdbc.duration_mean, trace.self.jdbc.duration_p95, trace.self.jdbc.row_count, and trace.self.jdbc.error_rate (each total_count=240 across all cmdb_ids in the window).\n"
    "- For the three containers of interest (docker_005, docker_006, docker_007), the JDBC groups appear with count=30 each. Notably:\n"
    "  • docker_005/006/007 — trace.self.jdbc.duration_p95: count=30, earliest=2020-04-11 04:30+08:00, max=0.011, global_p95=0.003, severity≈3.67x P95.\n"
    "  • jdbc.duration_mean and jdbc.row_count also elevated (duration_mean severity ≈2.0; row_count severity ≈1.05–1.09 for some containers).\n"
    "- No direct 'to_db'/'from_db' or explicit '.db.' style trace_name matches were found for these containers in the window; the docker traces show JDBC spans but do not name the target DB component.\n\n"
    "Conclusion / next step:\n"
    "- There are strong container-level JDBC anomalies (docker_005/006/007) coincident with the metric faults, but trace names do not explicitly identify which db_* host was targeted.\n"
    "- To tie container JDBC anomalies to specific DB components, inspect the JDBC spans' attributes (endpoint/peer/destination fields) or follow remoteprocess 'trace.from_' / 'trace.to_' edges from these containers to discover the target DB node(s). Prioritize docker_005/006/007 JDBC spans around 2020-04-11 04:30+08:00."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                       trace_name  total_count_in_window     earliest_timestamp_dt  max_value_in_window   global_p95  severity
36  trace.self.jdbc.duration_mean                    240 2020-04-11 04:30:00+08:00             0.077369     0.068958  1.121975
39      trace.self.jdbc.row_count                    240 2020-04-11 04:30:00+08:00          2483.000000  2369.600000  1.047856
37   trace.self.jdbc.duration_p95                    240 2020-04-11 04:30:00+08:00             0.100000     0.098977  1.010331
38     trace.self.jdbc.error_rate                    240 2020-04-11 04:30:00+08:00             0.000000     0.000000       NaN,        cmdb_id                     trace_name  count_in_window     earliest_timestamp_dt  max_value_in_window   global_p95  severity
1   docker_005   trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
5   docker_006   trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
9   docker_007   trace.self.jdbc.duration_p95               30 2020-04-11 04:30:00+08:00             0.011000     0.003000  3.666667
8   docker_007  trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005320     0.002627  2.025471
4   docker_006  trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005250     0.002621  2.003129
0   docker_005  trace.self.jdbc.duration_mean               30 2020-04-11 04:30:00+08:00             0.005134     0.002563  2.003043
3   docker_005      trace.self.jdbc.row_count               30 2020-04-11 04:30:00+08:00          1565.000000  1442.150000  1.085185
11  docker_007      trace.self.jdbc.row_count               30 2020-04-11 04:30:00+08:00          1467.000000  1426.950000  1.028067
7   docker_006      trace.self.jdbc.row_count               30 2020-04-11 04:30:00+08:00          1415.000000  1406.550000  1.006008
2   docker_005     trace.self.jdbc.error_rate               30 2020-04-11 04:30:00+08:00             0.000000     0.000000       NaN
6   docker_006     trace.self.jdbc.error_rate               30 2020-04-11 04:30:00+08:00             0.000000     0.000000       NaN
10  docker_007     trace.self.jdbc.error_rate               30 2020-04-11 04:30:00+08:00             0.000000     0.000000       NaN, Empty DataFrame
Columns: [cmdb_id, trace_name, count_in_window, earliest_timestamp_dt, max_value_in_window, global_p95, contains_db, severity]
Index: [], "Found trace_name groups referencing 'db' substrings: trace.self.jdbc.duration_mean (count=240); trace.self.jdbc.row_count (count=240); trace.self.jdbc.duration_p95 (count=240); trace.self.jdbc.error_rate (count=240)")```
```

In [ ]:
# In[11]:

import re
import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# Reuse existing trace_df, trace_thresholds, tz, start_local, end_local if present; otherwise load/compute
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace_thresholds
except NameError:
    trace_thresholds = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# Filter to incident window and target containers
containers = ['docker_005', 'docker_006', 'docker_007']
tw = trace_df[(trace_df['timestamp_dt'] >= start_local) & (trace_df['timestamp_dt'] <= end_local) & (trace_df['cmdb_id'].isin(containers))].copy()

# Regex to find edge-bearing trace_names and extract referenced component token
pattern = re.compile(r'\b(from|to)_((?:os|db|docker)_[0-9]+)\b', flags=re.IGNORECASE)
extracted = tw['trace_name'].str.extract(pattern)  # two columns: direction, referenced_component
extracted.columns = ['direction', 'referenced_component']
tw = tw.join(extracted)

# Keep only rows where a referenced_component was found
edges = tw[tw['referenced_component'].notna()].copy()

if not edges.empty:
    # Normalize referenced_component to lowercase (consistent token formatting)
    edges['referenced_component'] = edges['referenced_component'].str.lower()
    # Aggregate by (cmdb_id, referenced_component, trace_name)
    agg = edges.groupby(['cmdb_id','referenced_component','trace_name']).agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max')
    ).reset_index()
    # Join global_p95 from precomputed thresholds (match by cmdb_id + trace_name)
    agg = agg.merge(trace_thresholds[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
    # Compute severity = max_value_in_window / global_p95 (or inf if global_p95==0 and max>0)
    def compute_sev(r):
        p95 = r['global_p95']
        mv = r['max_value_in_window']
        if pd.isna(p95) or p95 == 0:
            return np.inf if (not pd.isna(mv) and mv > 0) else np.nan
        return mv / p95
    agg['severity'] = agg.apply(compute_sev, axis=1)
    # Sort by count desc and limit to top 20
    edges_display = agg.sort_values(['count_in_window','severity'], ascending=[False, False]).head(20).reset_index(drop=True)
    fallback_display = pd.DataFrame()  # empty fallback
else:
    # No edge-bearing trace_names found for these containers: return top 10 distinct trace_name values for these containers in window
    top_traces = tw.groupby('trace_name').agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min')
    ).reset_index().sort_values('count_in_window', ascending=False).head(10).reset_index(drop=True)
    edges_display = pd.DataFrame()  # empty edges
    fallback_display = top_traces

# Display compact results
edges_display, fallback_display

```
Out[11]:
```
Summary of cross-component trace edges for docker_005 / docker_006 / docker_007 (2020-04-11 04:30–05:00+08:00):

What was found
- There are multiple remoteprocess edge traces that explicitly reference other docker_* containers (patterns like trace.from_docker_X.remoteprocess.*). No direct traces referencing db_* components were found for these containers in the window.
- All edge-bearing groups start at 2020-04-11 04:30:00+08:00 and persist for the window (count_in_window = 30 for each group).

Top edge groups (sorted by count and severity) — examples:
- docker_006 <- docker_003:
  - trace.from_docker_003.remoteprocess.duration_p95 — count=30, earliest=04:30, max=0.06720, global_p95=0.04255, severity≈1.58
  - trace.from_docker_003.remoteprocess.duration_mean  — count=30, max=0.03752, global_p95≈0.02813, severity≈1.33
- docker_005 <- docker_004:
  - trace.from_docker_004.remoteprocess.duration_p95 — count=30, earliest=04:30, max=0.06770, global_p95=0.04444, severity≈1.52
  - trace.from_docker_004.remoteprocess.duration_mean — count=30, max=0.04151, global_p95=0.02920, severity≈1.42
- docker_005 <- docker_003:
  - trace.from_docker_003.remoteprocess.duration_p95 — count=30, earliest=04:30, max=0.06500, global_p95=0.04436, severity≈1.47
  - trace.from_docker_003.remoteprocess.duration_mean — count=30, max=0.03674, global_p95=0.02890, severity≈1.27
- docker_007 <- docker_001:
  - trace.from_docker_001.remoteprocess.duration_p95 — count=30, earliest=04:30, max=0.06500, global_p95=0.04564, severity≈1.42
  - trace.from_docker_001.remoteprocess.duration_mean — count=30, max=0.03924, global_p95=0.02756, severity≈1.42
- Several corresponding .row_count and .error_rate remoteprocess series also appear (count=30), with modest increases in row_count (severity ≈1.02–1.06) and error_rate usually at zero baseline.

Interpretation and recommended next steps
- The suspicious cross-component activity is between containers (docker_005/006/007 and docker_001/002/003/004), not directly naming any db_* hosts. This points to inter-container latency or load propagation rather than an immediately explicit DB-targeted trace.
- Because docker_005/006/007 also show strong JDBC anomalies at 04:30 (trace.self.jdbc.duration_p95 ≈3.7× P95), investigate these containers’ remoteprocess edges (especially the inbound edges listed above) and examine whether those upstream docker_* components in turn call actual DB hosts.
- Next concrete actions: inspect trace span attributes (peer/endpoint/destination) for the listed trace.from_docker_* spans on docker_005/006/007 at ~04:30 to find the ultimate DB peer(s); and inspect docker_003 and docker_004 (high-severity inbound edges) as potential upstream contributors.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id referenced_component                                         trace_name  count_in_window     earliest_timestamp_dt  max_value_in_window  global_p95  severity
0   docker_006           docker_003   trace.from_docker_003.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067200    0.042550  1.579318
1   docker_005           docker_004   trace.from_docker_004.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067700    0.044440  1.523402
2   docker_005           docker_003   trace.from_docker_003.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000    0.044357  1.465367
3   docker_007           docker_001   trace.from_docker_001.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000    0.045640  1.424189
4   docker_007           docker_001  trace.from_docker_001.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039240    0.027561  1.423769
5   docker_005           docker_004  trace.from_docker_004.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.041514    0.029196  1.421907
6   docker_007           docker_002   trace.from_docker_002.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.064000    0.045027  1.421354
7   docker_007           docker_002  trace.from_docker_002.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.038793    0.027519  1.409658
8   docker_006           docker_004  trace.from_docker_004.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039329    0.028636  1.373428
9   docker_006           docker_003  trace.from_docker_003.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.037523    0.028129  1.333976
10  docker_006           docker_004   trace.from_docker_004.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.073350    0.055560  1.320194
11  docker_005           docker_003  trace.from_docker_003.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.036743    0.028896  1.271545
12  docker_005           docker_003      trace.from_docker_003.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           327.000000  308.400000  1.060311
13  docker_007           docker_002      trace.from_docker_002.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           309.000000  295.050000  1.047280
14  docker_005           docker_004      trace.from_docker_004.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           322.000000  308.850000  1.042577
15  docker_007           docker_001      trace.from_docker_001.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           309.000000  300.350000  1.028800
16  docker_006           docker_003      trace.from_docker_003.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           290.000000  283.600000  1.022567
17  docker_006           docker_004      trace.from_docker_004.remoteprocess.row_count               30 2020-04-11 04:30:00+08:00           298.000000  292.850000  1.017586
18  docker_005           docker_003     trace.from_docker_003.remoteprocess.error_rate               30 2020-04-11 04:30:00+08:00             0.000000    0.000000       NaN
19  docker_005           docker_004     trace.from_docker_004.remoteprocess.error_rate               30 2020-04-11 04:30:00+08:00             0.000000    0.000000       NaN, Empty DataFrame
Columns: []
Index: [])```
```

In [ ]:
# In[12]:

import re
import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# Reuse existing trace_df and trace_thresholds; if missing, load/compute
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace_thresholds
except NameError:
    trace_thresholds = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        global_p95=lambda x: np.nanpercentile(x.values, 95),
        global_p05=lambda x: np.nanpercentile(x.values, 5),
        total_points='count'
    ).reset_index()

tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))

# Target upstream containers
upstream = ['docker_001','docker_002','docker_003','docker_004']

# Filter to incident window and upstream containers
tw = trace_df[
    (trace_df['timestamp_dt'] >= start_local) &
    (trace_df['timestamp_dt'] <= end_local) &
    (trace_df['cmdb_id'].isin(upstream))
].copy()

# 1) find trace_names containing the desired substrings (case-insensitive)
pattern_any = r'(to_|from_|remoteprocess|jdbc)'
mask_any = tw['trace_name'].str.contains(pattern_any, case=False, na=False)
tw_sel = tw[mask_any].copy()

# 2) extract referenced component tokens like to_docker_003, from_os_022, to_db_001, etc.
extract_pattern = re.compile(r'\b(?:to|from)_(?P<referenced_component>(?:docker|os|db)_[0-9]+)\b', flags=re.IGNORECASE)
extracted = tw_sel['trace_name'].str.extract(extract_pattern)
extracted.columns = ['referenced_component']
tw_sel = tw_sel.join(extracted)

# Keep only rows where a referenced component token was extracted
edges = tw_sel[tw_sel['referenced_component'].notna()].copy()
if not edges.empty:
    # normalize referenced_component to lowercase
    edges['referenced_component'] = edges['referenced_component'].str.lower()
    # 3) aggregate by (cmdb_id, referenced_component, trace_name)
    agg = edges.groupby(['cmdb_id','referenced_component','trace_name']).agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max')
    ).reset_index()
    # join global_p95 from full trace series per (cmdb_id, trace_name)
    agg = agg.merge(trace_thresholds[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
    # compute severity (max / global_p95) with inf handling
    def compute_sev(r):
        p95 = r['global_p95']
        mv = r['max_value_in_window']
        if pd.isna(p95) or p95 == 0:
            return np.inf if (not pd.isna(mv) and mv > 0) else np.nan
        return mv / p95
    agg['severity'] = agg.apply(compute_sev, axis=1)
    # sort and limit to top 20
    edges_display = agg.sort_values(['count_in_window','severity'], ascending=[False, False]).head(20).reset_index(drop=True)
    fallback_display = pd.DataFrame()  # empty fallback
else:
    # 4) fallback: top 10 distinct trace_name values for these containers in window
    top_traces = tw_sel.groupby('trace_name').agg(
        count_in_window=('value','size'),
        earliest_timestamp_dt=('timestamp_dt','min')
    ).reset_index().sort_values('count_in_window', ascending=False).head(10).reset_index(drop=True)
    edges_display = pd.DataFrame()  # empty
    fallback_display = top_traces

# One-line summary: whether any referenced components point to db_* or os_* (name up to 3)
if not edges.empty:
    refs = edges['referenced_component'].dropna().unique()
    db_refs = [r for r in refs if r.startswith('db_')]
    os_refs = [r for r in refs if r.startswith('os_')]
    found = []
    if db_refs:
        found.extend(db_refs[:3])
    if os_refs and len(found) < 3:
        found.extend(os_refs[: (3 - len(found)) ])
    if found:
        summary = "Upstream containers reference possible DB/OS hosts: " + ", ".join(found)
    else:
        summary = "Upstream containers reference other docker_* containers, no direct db_* or os_* referenced among top extracted tokens."
else:
    summary = "No edge-bearing trace_names with referenced components found for these upstream containers; returned top trace_name groups instead."

# Display compact outputs
edges_display, fallback_display, summary

```
Out[12]:
```
Summary (upstream containers docker_001–004, window 2020-04-11 04:30–05:00+08:00):

What was found
- Many cross-component remoteprocess edges and to_/from_ traces starting at 04:30 and persisting through the window. These show elevated latencies (severity >1) and point to inter-container communication, not direct db_* trace names.
- Notable upstream→referenced pairs (examples of top groups, count=30 each):
  - docker_001 ← os_022: trace.from_os_022.remoteprocess.duration_p95 — max 18.3785, global_p95 9.2301, severity ≈ 1.99.
  - docker_001 → docker_007: trace.to_docker_007.remoteprocess.duration_p95/mean — severity ≈ 1.42.
  - docker_003 → docker_006: trace.to_docker_006.remoteprocess.duration_p95 — severity ≈ 1.58.
  - docker_004 → docker_005: trace.to_docker_005.remoteprocess.duration_p95/mean — severity ≈ 1.52.
  - Several matching duration_mean/duration_p95 and row_count series also show modest increases (severity ≈1.2–1.6).

Do these upstream containers reference db_* or os_* that map to DB hosts?
- Yes — extracted referenced components include OS tokens (os_022, os_021) and many docker_* targets. No explicit db_* tokens were found among these upstream containers’ extracted references.
- Up to three relevant referenced components to inspect: os_022, os_021, and docker_006 (as an example inter-container hotspot).

Recommended next steps
- Inspect trace span attributes (peer/endpoint/DB host fields) for the listed from_os_022 spans on docker_001 and from_os_022 on docker_002 at ~04:30 to see if os_022 maps to a DB host.
- Follow the high-severity remoteprocess edges (docker_003→docker_006, docker_004→docker_005, docker_001→docker_007) to trace the call chain from containers with JDBC anomalies (docker_005/006/007) back to any OS/db nodes.
- Check os_022 and os_021 system metrics and DB connectivity logs around 04:30 as candidate nodes related to the incident.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id referenced_component                                       trace_name  count_in_window     earliest_timestamp_dt  max_value_in_window  global_p95  severity
0   docker_001               os_022     trace.from_os_022.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00            18.378500    9.230137  1.991140
1   docker_001               os_022    trace.from_os_022.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             8.146605    4.407002  1.848559
2   docker_003           docker_006   trace.to_docker_006.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067200    0.042550  1.579318
3   docker_004           docker_005   trace.to_docker_005.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.067700    0.044440  1.523402
4   docker_003           docker_005   trace.to_docker_005.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000    0.044357  1.465367
5   docker_001           docker_007   trace.to_docker_007.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.065000    0.045640  1.424189
6   docker_001           docker_007  trace.to_docker_007.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039240    0.027561  1.423769
7   docker_004           docker_005  trace.to_docker_005.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.041514    0.029196  1.421907
8   docker_002           docker_007   trace.to_docker_007.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.064000    0.045027  1.421354
9   docker_002           docker_008   trace.to_docker_008.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             3.837000    2.716217  1.412626
10  docker_002           docker_007  trace.to_docker_007.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.038793    0.027519  1.409658
11  docker_002               os_022    trace.from_os_022.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             6.890179    4.890980  1.408752
12  docker_004           docker_006  trace.to_docker_006.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.039329    0.028636  1.373428
13  docker_002               os_022     trace.from_os_022.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00            12.525100    9.180615  1.364299
14  docker_001           docker_008   trace.to_docker_008.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             3.689100    2.714450  1.359060
15  docker_002           docker_008  trace.to_docker_008.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             2.095112    1.548178  1.353276
16  docker_003           docker_006  trace.to_docker_006.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.037523    0.028129  1.333976
17  docker_004           docker_006   trace.to_docker_006.remoteprocess.duration_p95               30 2020-04-11 04:30:00+08:00             0.073350    0.055560  1.320194
18  docker_001           docker_008  trace.to_docker_008.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             2.141359    1.628216  1.315157
19  docker_003           docker_005  trace.to_docker_005.remoteprocess.duration_mean               30 2020-04-11 04:30:00+08:00             0.036743    0.028896  1.271545, Empty DataFrame
Columns: []
Index: [], 'Upstream containers reference possible DB/OS hosts: os_022, os_021')```
```

In [ ]:
# In[13]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse metric_df if available; otherwise load and prepare timestamp_dt
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P05 for every (cmdb_id=os_022, kpi_name) using the full metric_df (no filtering first)
thresholds_os022 = metric_df[metric_df['cmdb_id'] == 'os_022'].groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_p95=lambda x: np.nanpercentile(x.values, 95),
    global_p05=lambda x: np.nanpercentile(x.values, 5),
    total_points='count'
).reset_index()

# 2) Filter metric.csv to the incident window and restrict to os_022
start_local = tz.localize(datetime(2020, 4, 11, 4, 30, 0))
end_local = tz.localize(datetime(2020, 4, 11, 5, 0, 0))
metric_window_os022 = metric_df[
    (metric_df['timestamp_dt'] >= start_local) &
    (metric_df['timestamp_dt'] <= end_local) &
    (metric_df['cmdb_id'] == 'os_022')
].copy()

# 3) Mark anomalies per record by comparing to precomputed thresholds (join thresholds_os022)
metric_os22_th = metric_window_os022.merge(thresholds_os022[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')

def mark_anom_type(v, p95, p05):
    if pd.isna(v) or (pd.isna(p95) and pd.isna(p05)):
        return None
    if (not pd.isna(p95)) and (v >= p95):
        return 'high'
    if (not pd.isna(p05)) and (v <= p05):
        return 'low'
    return None

metric_os22_th['anomaly_type'] = metric_os22_th.apply(lambda r: mark_anom_type(r['value'], r['global_p95'], r['global_p05']), axis=1)
anoms_os22 = metric_os22_th[metric_os22_th['anomaly_type'].notnull()].copy()

# 4) Aggregate anomalies by kpi_name and anomaly_type
if anoms_os22.empty:
    agg_display = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_ratio'
    ])
else:
    agg = anoms_os22.groupby(['cmdb_id','kpi_name','anomaly_type']).agg(
        anomaly_count_in_window=('value','size'),
        earliest_anomaly_timestamp_dt=('timestamp_dt','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_p95=('global_p95','first'),
        global_p05=('global_p05','first')
    ).reset_index()

    # compute severity_ratio with division-by-zero handling
    def compute_severity(row):
        t = row['anomaly_type']
        if t == 'high':
            p95 = row['global_p95']
            maxv = row['max_value_in_window']
            if pd.isna(p95) or p95 == 0:
                return np.inf if (not pd.isna(maxv) and maxv > 0) else np.nan
            return maxv / p95
        else:  # low
            p05 = row['global_p05']
            minv = row['min_value_in_window']
            if pd.isna(minv) or minv == 0:
                return np.inf if (not pd.isna(p05) and p05 > 0) else np.nan
            return p05 / minv

    agg['severity_ratio'] = agg.apply(compute_severity, axis=1)

    # sort by anomaly_count desc then severity desc and limit to top 20
    agg_display = agg.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

    # select & order required columns
    agg_display = agg_display[[
        'cmdb_id','kpi_name','global_p95','global_p05','anomaly_type',
        'anomaly_count_in_window','earliest_anomaly_timestamp_dt',
        'max_value_in_window','min_value_in_window','severity_ratio'
    ]]

# 5) One-line recommendation: top 1-2 os_022 KPIs by anomaly_count then severity
if agg_display.shape[0] == 0:
    recommendation = "No anomalies detected for os_022 in the incident window."
else:
    top = agg_display.sort_values(['anomaly_count_in_window','severity_ratio'], ascending=[False, False]).head(2)
    parts = []
    for _, r in top.iterrows():
        parts.append(f"{r['kpi_name']} ({r['anomaly_type']}, count={int(r['anomaly_count_in_window'])}, sev={r['severity_ratio'] if not pd.isna(r['severity_ratio']) else 'nan'})")
    recommendation = "Recommend inspect os_022 KPIs: " + "; ".join(parts)

# Final compact outputs
agg_display, recommendation

```
Out[13]:
```
```python
summary = (
    "os_022 anomaly summary (2020-04-11 04:30–05:00 UTC+8):\n\n"
    "- Strongest sustained anomalies (start 04:30, 30 points):\n"
    "  * Disk_await — type: high, count=30, max=0.300, global_P95=0.0 → severity=inf (non-zero latency above zero baseline).\n"
    "  * Agent_ping — type: high, count=30, value=1.0, global_P95=1.0 → severity=1.0 (at-baseline ping).\n"
    "  * ICMP_ping — type: high, count=30, value=1.0, global_P95=1.0 → severity=1.0.\n\n"
    "- Other notable flags: several disk-related KPIs (Disk_io_util, Disk_rd_ios, Disk_rd_kbs, Page_pi) are flagged high but have global_P95=0 and max=0 (flags from zero baselines). Disk_wr_ios shows both high and low segments (15 points each, values 4–5). Disk_svctm shows low flags (27 points).\n\n"
    "Interpretation:\n"
    "- The clearest actionable signal is elevated disk latency (Disk_await) on os_022 beginning at 04:30. Many other KPIs are flagged due to zero baselines and need semantic checks.\n\n"
    "Recommendation:\n"
    "- First inspect os_022 disk IO/latency (Disk_await) around 2020-04-11 04:30+08:00. Next check network/agent connectivity (Agent_ping / ICMP_ping) and then disk service metrics (Disk_svctm, Disk_wr_ios) to determine whether IO or network issues caused the container-level JDBC and remoteprocess latency seen on docker_005/006/007."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                   kpi_name  global_p95  global_p05 anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp_dt  max_value_in_window  min_value_in_window  severity_ratio
0   os_022                 Disk_await      0.0000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.300                0.000             inf
1   os_022                 Agent_ping      1.0000       1.000         high                       30     2020-04-11 04:30:00+08:00                1.000                1.000             1.0
2   os_022                  ICMP_ping      1.0000       1.000         high                       30     2020-04-11 04:30:00+08:00                1.000                1.000             1.0
3   os_022               Disk_io_util      0.0000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
4   os_022                Disk_rd_ios      0.0000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
5   os_022                Disk_rd_kbs      0.0000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
6   os_022                    Page_pi      0.0000       0.000         high                       30     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
7   os_022                 Disk_svctm      0.9445       0.000          low                       27     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
8   os_022                Disk_wr_ios      5.0000       4.000         high                       15     2020-04-11 04:30:00+08:00                5.000                5.000             1.0
9   os_022                Disk_wr_ios      5.0000       4.000          low                       15     2020-04-11 04:32:00+08:00                4.000                4.000             1.0
10  os_022                FS_max_util     89.0000      89.000         high                        6     2020-04-11 04:30:00+08:00               89.000               89.000             1.0
11  os_022             FS_total_space    110.1960     110.196         high                        6     2020-04-11 04:30:00+08:00              110.196              110.196             1.0
12  os_022    Received_errors_packets      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
13  os_022             Received_queue      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
14  os_022        Sent_errors_packets      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
15  os_022              Shared_memory      0.0000       0.000         high                        6     2020-04-11 04:30:00+08:00                0.000                0.000             NaN
16  os_022              Swap_used_pct      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
17  os_022  System_block_queue_length      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
18  os_022             Zombie_Process      0.0000       0.000         high                        6     2020-04-11 04:34:00+08:00                0.000                0.000             NaN
19  os_022   Num_of_running_processes      2.0000       1.000          low                        5     2020-04-11 04:34:00+08:00                1.000                1.000             1.0, 'Recommend inspect os_022 KPIs: Disk_await (high, count=30, sev=inf); Agent_ping (high, count=30, sev=1.0)')```
```